# GraviTree on COMPAS and FICO

Fit a Rashomon set of falling trees on COMPAS and FICO. `binarize=True` turns continuous features into threshold features during `fit`.  

Note that the model runs slower on Jupyter as compared to native Python. 


In [1]:
from pathlib import Path
import time

import pandas as pd
from sklearn.model_selection import train_test_split

from falling_trees.gravitree import GraviTree

ROOT = Path.cwd()
if not (ROOT / "data" / "benchmark" / "compas.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "benchmark"

def load_split(csv_name, label, test_size=0.2, seed=0):
    frame = pd.read_csv(DATA / csv_name)
    train, test = train_test_split(
        frame, test_size=test_size, random_state=seed, stratify=frame[label]
    )
    return (
        train.drop(columns=[label]),
        train[label].astype(int),
        test.drop(columns=[label]),
        test[label].astype(int),
    )

def fit_and_print(csv_name, label, dataset_name):
    X_train, y_train, X_test, y_test = load_split(csv_name, label)
    model = GraviTree(
        depth=5,
        eps=0.01,
        min_support=0.05,
        branching_cost=0.01,
        binarize=True,
        num_estimators=70,
    )
    model.fit(X_train, y_train)
    scores = [model.score(X_train, y_train, tree_idx=i) for i in range(len(model.ft_rset))] # scores are ordered by accuracy
    index = max(range(len(scores)), key=scores.__getitem__)
    test_acc = model.score(X_test, y_test, tree_idx=index)
    print(
        f"{dataset_name}: {len(model.ft_rset)} trees, "
        f"{len(model.feature_names_)} binary features, "
        f"{len(y_train)} train / {len(y_test)} test rows"
    )
    print(
        f"Most accurate tree is index {index}: "
        f"train accuracy {scores[index]:.3f}, test accuracy {test_acc:.3f}"
    )
    model.print_tree(tree_idx=index)
    return model



## COMPAS

In [2]:
start = time.perf_counter()
compas = fit_and_print('compas.csv', 'two_year_recid', 'COMPAS')
end = time.perf_counter()
print("Took {} seconds!".format(end-start))

COMPAS: 501 trees, 28 binary features, 5525 train / 1382 test rows
Most accurate tree is index 0: train accuracy 0.685, test accuracy 0.658
if priors_count<=1.5:
    if age<=21.5:
        return 0.6075581395348837
    else:
        if age<=30.5:
            return 0.33998100664767333
        else:
            return 0.23152337858220212
else:
    if priors_count<=6.5:
        if age<=32.5:
            return 0.6529814271749755
        else:
            return 0.38549618320610685
    else:
        return 0.7170191339375629
Took 29.64318084716797 seconds!


## FICO

In [3]:
start = time.perf_counter()
fico = fit_and_print('fico.csv', 'RiskPerformance', 'FICO')
end = time.perf_counter()
print("Took {} seconds!".format(end-start))

FICO: 106 trees, 31 binary features, 8367 train / 2092 test rows
Most accurate tree is index 0: train accuracy 0.712, test accuracy 0.707
if ExternalRiskEstimate<=81.5:
    if ExternalRiskEstimate<=70.5:
        if PercentInstallTrades<=45.5:
            return 0.3159121171770972
        else:
            return 0.1793041926851026
    else:
        if NumSatisfactoryTrades<=27.5:
            if MSinceMostRecentInqexcl7days<=0.5:
                if AverageMInFile<=74.5:
                    return 0.39663093415007655
                else:
                    return 0.5704809286898839
            else:
                return 0.6466431095406361
        else:
            return 0.7157360406091371
else:
    return 0.8069852941176471
Took 127.78324737213552 seconds!


In [4]:
fico.print_tree(tree_idx=70)

if MSinceMostRecentInqexcl7days<=0.5:
    if ExternalRiskEstimate<=78.5:
        if ExternalRiskEstimate<=74.5:
            return 0.2841599190283401
        else:
            return 0.5616666666666666
    else:
        return 0.7338376891334251
else:
    if ExternalRiskEstimate<=81.5:
        if ExternalRiskEstimate<=67.5:
            return 0.32585949177877427
        else:
            return 0.676094890510949
    else:
        return 0.8624161073825504
